# Laboratorio 3 - Clasificación
### Sergio David Ferreira - Juan David Gutierrez

In [ ]:
import warnings
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import add_constant


In [ ]:
data = pd.read_csv('data/Datos_Laboratorio3.csv', sep=';')
data.describe()

In [ ]:
X = data.drop(columns=['Plan_entrenamiento', 'Plan_nutricion'])
y_entrenamiento = data['Plan_entrenamiento']
y_nutricion = data['Plan_nutricion']
X_train, X_test, y_entrenamiento_train, y_entrenamiento_test, y_nutricion_train, y_nutricion_test = train_test_split(
    X, y_entrenamiento, y_nutricion, test_size=0.2, random_state=42)

In [ ]:
((X.isnull().sum()/X.shape[0])).sort_values(ascending=False)

In [ ]:
categorical_cols = X.select_dtypes(include=['str']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
ordinal_cols = ['Nivel_Actividad', 'Nivel_experiencia']
binary_cols = ['Tiene_alergia', 'Problemas_digestivos', 'Fumador', 'Alcohol']

categorical_cols = list(categorical_cols.drop(ordinal_cols))
numerical_cols = list(numerical_cols.drop(binary_cols))
encoder = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('ordinal', OrdinalEncoder(categories=[['Bajo', 'Moderado', 'Alto'], ['Principiante', 'Intermedio', 'Avanzado']]), ordinal_cols)
])

In [ ]:
X[numerical_cols].hist(figsize=(10, 10))

In [ ]:
def limiter(data):
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data, columns=numerical_cols + binary_cols)
    data = data.copy()
    data['Horas_sueño'] = data['Horas_sueño'].clip(4, 15)
    return data

limiter_transformer = FunctionTransformer(limiter)


In [ ]:
X_vif = add_constant(X[numerical_cols].dropna())
vif = pd.DataFrame()
vif['Variable'] = X_vif.columns
vif['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif[1:].sort_values('VIF', ascending=False))

In [ ]:
X_vif = add_constant(X[numerical_cols].dropna())
X_vif.drop(columns=['Peso'], inplace=True)
vif = pd.DataFrame()
vif['Variable'] = X_vif.columns
vif['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif[1:].sort_values('VIF', ascending=False))

In [ ]:
def dropper(data):
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data, columns=numerical_cols + binary_cols)
    return data.drop(columns=['Alcohol', 'Fumador', 'Peso'], errors='ignore')

dropper_transformer = FunctionTransformer(dropper)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('limiter', limiter_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('dropper', dropper_transformer),
])


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('categorical', encoder, categorical_cols + ordinal_cols),
    ('numerical', numeric_transformer, numerical_cols + binary_cols),
])
preprocessor

## Modelos de regresion logistica con busqueda de hiperparametros
Se entrenan dos modelos (entrenamiento y nutricion) usando el mismo preprocesamiento dentro de un `Pipeline` y `GridSearchCV` con validacion cruzada.


In [ ]:
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn.linear_model._logistic')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Nota: no se usa 'liblinear' porque en esta version no soporta multiclase (3 o mas clases).
param_grid_lr = [
    {
        'classifier__solver': ['lbfgs', 'newton-cg', 'saga'],
        'classifier__penalty': ['l2'],
        'classifier__C': [0.1, 1, 10],
        'classifier__class_weight': [None, 'balanced'],
    },
    {
        'classifier__solver': ['saga'],
        'classifier__l1_ratio': [0.1, 0.5, 0.9],
        'classifier__C': [0.1, 1, 10],
        'classifier__class_weight': [None, 'balanced'],
    },
]

def run_logistic_gridsearch(target_name, y_train, y_test):
    pipeline_lr = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=5000, random_state=42)),
    ])

    grid = GridSearchCV(
        estimator=pipeline_lr,
        param_grid=param_grid_lr,
        scoring='f1_macro',
        cv=cv,
        n_jobs=1,
        refit=True,
        verbose=1,
        error_score='raise',
    )

    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)

    print(f'\n{target_name}')
    print('-' * len(target_name))
    print('Mejores hiperparametros:', grid.best_params_)
    print('Mejor F1-macro (CV):', round(grid.best_score_, 4))
    print('Accuracy (test):', round(accuracy_score(y_test, y_pred), 4))
    print('F1-weighted (test):', round(f1_score(y_test, y_pred, average='weighted'), 4))
    print('F1-macro (test):', round(f1_score(y_test, y_pred, average='macro'), 4))
    print('\nReporte de clasificacion:\n')
    print(classification_report(y_test, y_pred))

    cv_summary = (
        pd.DataFrame(grid.cv_results_)[['mean_test_score', 'std_test_score', 'params']]
        .sort_values('mean_test_score', ascending=False)
        .head(10)
    )
    print('Top 10 combinaciones por F1-macro en CV:')
    print(cv_summary.to_string(index=False))

    return grid


In [ ]:
grid_entrenamiento = run_logistic_gridsearch(
    target_name='Modelo para recomendacion de plan de entrenamiento',
    y_train=y_entrenamiento_train,
    y_test=y_entrenamiento_test,
)


In [ ]:
grid_nutricion = run_logistic_gridsearch(
    target_name='Modelo para recomendacion de plan de nutricion',
    y_train=y_nutricion_train,
    y_test=y_nutricion_test,
)
